# Install openAI

In [38]:
# pip install openai

In [5]:
import openai

print(openai.__version__)

2.32.0


In [ ]:
# Ideally you should put this in environment or .env file

OPENAI_API_KEY = "sk-proj-FlhuSo012xZfdkfdfgjlhfdsklfhhhhhhhhhhflskdhfdfsluuuuuuudfgjlfgdkfdlgkdfgkgkdgkdfgkl[fgkl[fpYA"

# Input files: training + validation
- training has 20 records
- validation has 10 records

In [3]:
# Content of training file

with open("training-medical-records.csv", 'r') as file:
    for i in range(4):
        line = file.readline()
        print(line.strip()) # strip() removes extra newlines
        print("-----------------")


medical_report,extracted_json
-----------------
"Nancy Johnson, a 45-year-old female, presented to the clinic with complaints of severe migraine headaches occurring 3-4 times per week for the past 3 months. Patient reports throbbing pain on the right side of head, accompanied by photophobia and nausea. No previous history of migraines. Family history positive for migraines (mother). Physical examination revealed normal neurological findings. Blood pressure 118/76 mmHg. Prescribed Sumatriptan 50mg for acute episodes, with instructions to take at onset of symptoms.","{""patient name"": ""Nancy Johnson"", ""age"": 45, ""diagnosis"": ""migraine headaches"", ""prescribed medication"": ""Sumatriptan""}"
-----------------
"Mike Chen, a 58-year-old male with a 10-year history of smoking (quit 5 years ago), presented for routine check-up. Blood pressure readings consistently elevated: 158/95 mmHg. Patient reports occasional headaches and fatigue. Family history significant for cardiovascular di

# Prepare data

In [11]:
# Here we convert the data to format 
# that open ai wants

import csv
import json

openai_training_file   = "training_data_TEMP.jsonl"
openai_validation_file = "validation_data_TEMP.jsonl"

def convert_csv_to_training_format(input_csv, output_file):
    system_message = {
        "role": "system",
        "content": "Extract Details from medical report"
    }

    with open(input_csv, 'r', encoding='utf-8') as csvfile, \
         open(output_file, 'w', encoding='utf-8') as outfile:
        reader = csv.reader(csvfile)
        next(reader)  # Skip header

        for row in reader:
            medical_report = row[0]
            extracted_json = row[1]

            training_example = {
                "messages": [
                    system_message,
                    {"role": "user", "content": medical_report},
                    {"role": "assistant", "content": extracted_json}
                ]
            }
            outfile.write(json.dumps(training_example) + '\n')


convert_csv_to_training_format("training-medical-records.csv", openai_training_file)
convert_csv_to_training_format("validation-medical-records.csv", openai_validation_file)

print("Training data prepared successfully!")

Training data prepared successfully!


In [12]:
# Lets look the structure and content of file

with open(openai_training_file, 'r') as file:
    for i in range(3):
        line = file.readline()
        if not line: break  # Stop if the file has fewer than 3 lines
        print(line.strip()) # strip() removes extra newlines
        print("-----------------")


{"messages": [{"role": "system", "content": "Extract Details from medical report"}, {"role": "user", "content": "Nancy Johnson, a 45-year-old female, presented to the clinic with complaints of severe migraine headaches occurring 3-4 times per week for the past 3 months. Patient reports throbbing pain on the right side of head, accompanied by photophobia and nausea. No previous history of migraines. Family history positive for migraines (mother). Physical examination revealed normal neurological findings. Blood pressure 118/76 mmHg. Prescribed Sumatriptan 50mg for acute episodes, with instructions to take at onset of symptoms."}, {"role": "assistant", "content": "{\"patient name\": \"Nancy Johnson\", \"age\": 45, \"diagnosis\": \"migraine headaches\", \"prescribed medication\": \"Sumatriptan\"}"}]}
-----------------
{"messages": [{"role": "system", "content": "Extract Details from medical report"}, {"role": "user", "content": "Mike Chen, a 58-year-old male with a 10-year history of smok

# Get the model that would be fine tuned from the website

In [14]:
import os
from openai import OpenAI
from time import sleep

MODEL = "gpt-4o-mini-2024-07-18" # Get the name from website

# Initialize OpenAI client
client = OpenAI(api_key = OPENAI_API_KEY)

In [15]:
# Check if API key is working
try:
    response = client.chat.completions.create(
        model=MODEL, # Use a mini model to save costs
        messages=[{"role": "user", "content": "hi"}],
        max_tokens=1
    )
    print("API Key and Chat Completion working.")
    
except Exception as e:
    print(f"Error: {e}")


API Key and Chat Completion working.


# Upload training + validation file and get file id

In [16]:
def upload_file(file_path):
    """Upload training file to OpenAI"""
    
    with open(file_path, "rb") as file:
        response = client.files.create(
            file=file,
            purpose="fine-tune"
        )
        return response.id

In [17]:
training_file_id = upload_file(openai_training_file)
validation_file_id = upload_file(openai_validation_file)

(training_file_id,validation_file_id

('file-Qv1zsCvFzy9CTQFVajw6qS', 'file-2mNXT64yW5zaUVJFkz6veM')

# Create fine tuning job

In [18]:
def create_fine_tuning_job(training_file_id, validation_file_id=None, model=MODEL):
    """Create a fine-tuning job"""
    
    response = client.fine_tuning.jobs.create(
        training_file=training_file_id,
        validation_file=validation_file_id,
        model=model
    )
    
    return response.id

In [19]:
job_id = create_fine_tuning_job(training_file_id, validation_file_id, MODEL)
job_id

'ftjob-zqNR7MhRjfXJWK18r4GShMNi'

# Run + Monitor: (Took 17 minutes)

In [4]:
def monitor_job(job_id):
    """Monitor fine-tuning job progress"""
    
    while True:
        job = client.fine_tuning.jobs.retrieve(job_id)
        print(f"Status: {job.status}")

        if job.status in ["succeeded", "failed"]:
            return job

        # List latest events
        events = client.fine_tuning.jobs.list_events(
            fine_tuning_job_id=job_id,
            limit=5
        )
        for event in events.data:
            print(f"Event: {event.message}")

        sleep(20)  # Check every 30 seconds

In [21]:
# Took 17 minutes
job =  monitor_job(job_id)

if job.status == "succeeded":
  fine_tuned_model = job.fine_tuned_model
  print(f"Fine-tuned model ID: {fine_tuned_model}")
    
else:
  print("Fine-tuning failed.")

Status: validating_files
Event: Validating training file: file-Qv1zsCvFzy9CTQFVajw6qS and validation file: file-2mNXT64yW5zaUVJFkz6veM
Event: Created fine-tuning job: ftjob-zqNR7MhRjfXJWK18r4GShMNi
Status: validating_files
Event: Validating training file: file-Qv1zsCvFzy9CTQFVajw6qS and validation file: file-2mNXT64yW5zaUVJFkz6veM
Event: Created fine-tuning job: ftjob-zqNR7MhRjfXJWK18r4GShMNi
Status: validating_files
Event: Validating training file: file-Qv1zsCvFzy9CTQFVajw6qS and validation file: file-2mNXT64yW5zaUVJFkz6veM
Event: Created fine-tuning job: ftjob-zqNR7MhRjfXJWK18r4GShMNi
Status: running
Event: Fine-tuning job started
Event: Files validated, moving job to queued state
Event: Validating training file: file-Qv1zsCvFzy9CTQFVajw6qS and validation file: file-2mNXT64yW5zaUVJFkz6veM
Event: Created fine-tuning job: ftjob-zqNR7MhRjfXJWK18r4GShMNi
Status: running
Event: Fine-tuning job started
Event: Files validated, moving job to queued state
Event: Validating training file: file

In [22]:
job = client.fine_tuning.jobs.retrieve(job_id)

print(job)

FineTuningJob(id='ftjob-zqNR7MhRjfXJWK18r4GShMNi', created_at=1777096152, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4o-mini-2024-07-18:personal::DYQRqZPg', finished_at=1777097188, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=1.8, n_epochs=5), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-gNFgkjhojxOGjISHDeT2ngsr', result_files=['file-Krtycr7PcdD9yqHgZnUv83'], seed=1546642327, status='succeeded', trained_tokens=11205, training_file='file-Qv1zsCvFzy9CTQFVajw6qS', validation_file='file-2mNXT64yW5zaUVJFkz6veM', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=1.8, n_epochs=5))), user_provided_suffix=None, usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=N

In [32]:

print(f"Status: {job.status}")
print(f"Model: {job.fine_tuned_model}")
print(f"Tokens used: {job.trained_tokens}")
print(f"Training file: {job.training_file}")
print(f"Validation file: {job.validation_file}")
print(f"hyperparameters : {job.hyperparameters}")

print(f"Time taken (seconds): {job.finished_at - job.created_at if job.finished_at else 'Running'}")


Status: succeeded
Model: ft:gpt-4o-mini-2024-07-18:personal::DYQRqZPg
Tokens used: 11205
Training file: file-Qv1zsCvFzy9CTQFVajw6qS
Validation file: file-2mNXT64yW5zaUVJFkz6veM
hyperparameters : Hyperparameters(batch_size=1, learning_rate_multiplier=1.8, n_epochs=5)
Time taken (seconds): 1036


In [37]:
# time in  minutes
1036/60

17.266666666666666

# Go to Open AI and view the metrics on fine tuning dashboard

# Test model

In [33]:
def test_model(model_id, test_input):
    """Test the fine-tuned model"""
    
    completion = client.chat.completions.create(
        model=model_id,
        messages=[
            {
                "role": "system",
                "content": ""   # NOTE: No instruction is provided here
            },
            {"role": "user", "content": test_input}
        ]
    )
    return completion.choices[0].message


In [34]:
patient_data = """Loxford Kumar, a 19-year-old male, presents with severe acne on face and 
upper back present for 1 year. He works at Loxford Academy. Multiple inflammatory papules and nodules noted on examination. 
Previous trials of over-the-counter treatments ineffective. Started on Isotretinoin 40mg daily 
with monthly liver function monitoring."""

result = test_model(fine_tuned_model, patient_data)

print(result)

ChatCompletionMessage(content='{"patient name": "Loxford Kumar", "age": 19, "diagnosis": "severe acne", "prescribed medication": "Isotretinoin"}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [35]:
print(result.content)

{"patient name": "Loxford Kumar", "age": 19, "diagnosis": "severe acne", "prescribed medication": "Isotretinoin"}


In [36]:
import json

print(json.loads(result.content))

{'patient name': 'Loxford Kumar', 'age': 19, 'diagnosis': 'severe acne', 'prescribed medication': 'Isotretinoin'}


# You can also run this chat via fine-tuning -> playground dashboard

# How students/others can access this fine tuned model ?

In [42]:
# Name of the model

fine_tuned_model

'ft:gpt-4o-mini-2024-07-18:personal::DYQRqZPg'

In [43]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY) # This is my own API key. The students API key is not going to work


user_content = """Loxford Kumar, a 19-year-old male, presents with severe acne on face and 
upper back present for 1 year. He works at Loxford Academy. Multiple inflammatory papules and nodules noted on examination. 
Previous trials of over-the-counter treatments ineffective. Started on Isotretinoin 40mg daily 
with monthly liver function monitoring."""

response = client.chat.completions.create(
    model="ft:gpt-4o-mini-2024-07-18:personal::DYQRqZPg",  # your fine-tuned model
    messages=[
        {"role": "system", "content": ""}, # NOTE: Nothing in the content
        {"role": "user", "content": user_content}
    ]
)

print(response.choices[0].message.content)


{"patient name": "Loxford Kumar", "age": 19, "diagnosis": "severe acne", "prescribed medication": "Isotretinoin"}


# Few important points:

Question: Does it cost money to keep it?

❌ No storage cost


**You only pay for its usage.**

# To delete: 
### client.models.delete(fine_tuned_model)